
# 🚀 MEGA Financial Fraud Intelligence System (All-in-One)

Includes:

- Synthetic fraud dataset
- Imbalance handling (SMOTE)
- AutoML model selection
- Ensemble stacking
- XGBoost
- Isolation Forest (behavioral anomaly)
- Graph fraud detection (NetworkX)
- SHAP explainability
- Threshold optimization
- Concept drift detection
- Business impact simulation


In [ ]:

!pip install pandas numpy scikit-learn matplotlib imbalanced-learn xgboost shap networkx scipy joblib


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier, IsolationForest, StackingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap
import networkx as nx
from scipy.stats import ks_2samp
np.random.seed(42)


## 1️⃣ Generate Imbalanced Dataset

In [ ]:

X = np.random.normal(0,1,(6000,15))
y = np.random.choice([0,1],6000,p=[0.97,0.03])

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)
print("Fraud ratio:",sum(y)/len(y))


## 2️⃣ Handle Imbalance (SMOTE)

In [ ]:

sm = SMOTE()
X_train_sm,y_train_sm = sm.fit_resample(X_train,y_train)
print("New Fraud ratio:",sum(y_train_sm)/len(y_train_sm))


## 3️⃣ AutoML Model Selection

In [ ]:

models = {
    "Logistic": LogisticRegression(),
    "RandomForest": RandomForestClassifier(),
    "XGBoost": XGBClassifier(eval_metric='logloss')
}

best_model = None
best_auc = 0

for name, model in models.items():
    model.fit(X_train_sm,y_train_sm)
    probs = model.predict_proba(X_test)[:,1]
    fpr,tpr,_ = roc_curve(y_test,probs)
    score = auc(fpr,tpr)
    print(name,"AUC:",score)
    if score > best_auc:
        best_auc = score
        best_model = model

print("Best Model Selected with AUC:",best_auc)


## 4️⃣ Ensemble Stacking

In [ ]:

estimators = [
    ('rf', RandomForestClassifier()),
    ('xgb', XGBClassifier(eval_metric='logloss'))
]

stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
stack.fit(X_train_sm,y_train_sm)

stack_probs = stack.predict_proba(X_test)[:,1]
stack_auc = auc(*roc_curve(y_test,stack_probs)[:2])

print("Stacking AUC:",stack_auc)


## 5️⃣ ROC Curve

In [ ]:

fpr,tpr,_ = roc_curve(y_test,stack_probs)

plt.figure()
plt.plot(fpr,tpr)
plt.title("Stacking ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()


## 6️⃣ Behavioral Anomaly Detection

In [ ]:

iso = IsolationForest()
iso.fit(X_train)

anomaly_scores = iso.decision_function(X_test)

plt.figure()
plt.hist(anomaly_scores,bins=30)
plt.title("Anomaly Score Distribution")
plt.show()


## 7️⃣ Graph-Based Fraud Detection

In [ ]:

G = nx.erdos_renyi_graph(100,0.05)
centrality = nx.degree_centrality(G)

plt.figure()
plt.hist(list(centrality.values()),bins=20)
plt.title("Network Centrality Distribution")
plt.show()


## 8️⃣ SHAP Explainability

In [ ]:

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test[:200])

shap.summary_plot(shap_values,X_test[:200])


## 9️⃣ Threshold Optimization

In [ ]:

thresholds = np.arange(0.1,0.9,0.05)
counts=[]

for t in thresholds:
    preds = (stack_probs>=t).astype(int)
    counts.append(sum(preds))

plt.figure()
plt.plot(thresholds,counts)
plt.title("Alerts vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Fraud Alerts")
plt.show()


## 🔟 Concept Drift Detection

In [ ]:

new_data = np.random.normal(0.5,1,(1000,15))
stat,p = ks_2samp(X_test[:,0],new_data[:,0])

print("KS Statistic:",stat)
print("Drift Detected?" , p<0.05)


## 1️⃣1️⃣ Business Impact Simulation

In [ ]:

threshold = 0.5
preds = (stack_probs>=threshold).astype(int)

total = len(preds)
flagged = sum(preds)
reduction = (1-flagged/total)*100

print("Total Transactions:",total)
print("Flagged Fraud:",flagged)
print("Workload Reduction %:",reduction)
